# Estudos Dirigidos 1 - TCL e IC em Distribuições Assimétricas

Este exercício prático visa demonstrar numericamente a validade do Teorema Central do Limite (TCL) e a precisão dos Intervalos de Confiança (IC) baseados na distribuição t, mesmo quando a população original segue uma distribuição assimétrica, como a Exponencial.

# Parte I: Simulação do TCL, IC e Distribuições Assimétricas

## 1.1 Exercício: TCL em População Assimétrica (Exponencial)

**Objetivo:**  
Simular o **Teorema Central do Limite (TCL)** para uma população com distribuição **Exponencial** (\(λ = 0.2, μ = 5\)) e verificar a **taxa de cobertura** do Intervalo de Confiança (IC) para \(n\) grande.

### Parâmetros da Simulação
- **População gerada:** Distribuição Exponencial, \(μ = 5\)  
- **Tamanhos amostrais:**  
  - \(n = 5\) (pequena)  
  - \(n = 50\) (grande)  
- **Número de simulações:** \(k = 10.000\)



### 1.1.1 Código em Python: Simulação do TCL e Cobertura do IC

Implemente a simulação:
Em ambos os casos, deve gerar a população, coletar as 10.000 médias amostrais para n = 5 e n = 50, e gerar os histogramas para comparação.

In [ ]:
import numpy as np
from scipy.stats import t, skew

TRUE_MEAN = 5
RATE = 0.2
POP_SIZE = 100_000
NUM_SIMULATIONS = 10_000
SAMPLE_SIZE = 50
CONFIDENCE_LEVEL = 0.95

np.random.seed(1)
population = np.random.exponential(scale=1/RATE, size=POP_SIZE)
ic_contains_mean = 0

# 1. Simular TCL (Assimetria)
# print(f"Assimetria da Populacao: {skew(population):.4f}")
# meios_n50 = [np.mean(np.random.choice(population, SAMPLE_SIZE)) for _ in range(NUM_SIMULATIONS)]
# print(f"Assimetria Dist. Medias (n=50): {skew(meios_n50):.4f}")

# 2. Simular Cobertura do IC
for _ in range(NUM_SIMULATIONS):
    sample = np.random.choice(population, SAMPLE_SIZE, replace=False)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    standard_error = sample_std / np.sqrt(SAMPLE_SIZE)
    degrees_freedom = SAMPLE_SIZE - 1
    ic_lower, ic_upper = t.interval(
        confidence=CONFIDENCE_LEVEL,
        df=degrees_freedom,
        loc=sample_mean,
        scale=standard_error
    )
    if (ic_lower <= TRUE_MEAN) and (TRUE_MEAN <= ic_upper):
        ic_contains_mean += 1

proportion_success = ic_contains_mean / NUM_SIMULATIONS
print(f"Taxa de Cobertura do IC (Esperada 0.95): {proportion_success:.4f}")

# Parte II: Exercícios de ANOVA e Planejamento Experimental

## 2.1 Exercício 1: ANOVA de Fator Único (One-Way) com Tukey HSD

**Cenário:**  
Comparação do tempo de secagem de três revestimentos (**A**, **B** e **C**), cada um com $n = 10$ repetições.  
Hipótese esperada: $\mu_A \approx \mu_B \neq \mu_C$.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

np.random.seed(123)
n_rep = 10
# Definindo as médias: A=20, B=21 (similares), C=28 (diferente)
revestimento = np.repeat(['A', 'B', 'C'], n_rep)
media_A = 20 + np.random.normal(0, 1.5, n_rep)
media_B = 21 + np.random.normal(0, 1.5, n_rep)
media_C = 28 + np.random.normal(0, 1.5, n_rep)
tempo_secagem = np.concatenate([media_A, media_B, media_C])

dados_one_way = pd.DataFrame({
    'Revestimento': revestimento,
    'Tempo_Secagem': tempo_secagem
})

# Ajuste do Modelo ANOVA
modelo_one_way = ols('Tempo_Secagem ~ C(Revestimento)', data=dados_one_way).fit()
resultado_anova = sm.stats.anova_lm(modelo_one_way, typ=2)
print(resultado_anova)

# Teste Post-Hoc de Tukey HSD
resultado_tukey = pairwise_tukeyhsd(endog=dados_one_way['Tempo_Secagem'],
                                    groups=dados_one_way['Revestimento'],
                                    alpha=0.95)
print(resultado_tukey)

## 2.2 Exercício 2: Delineamento em Blocos Casualizados (RCBD)

**Cenário:**
Teste de 4 dietas (D1-D4) em 3 gaiolas (G1-G3) como blocos. Forte efeito de bloco (Gaiola) e efeito de tratamento (D4 superior).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

np.random.seed(456)
# 4 dietas x 3 blocos
tratamento = np.tile(['D1', 'D2', 'D3', 'D4'], 3)
bloco = np.repeat(['G1', 'G2', 'G3'], 4)

# Efeitos: Bloco (G1=0, G2=5, G3=10); Tratamento (D1,D2,D3=0, D4=8)
efeito_bloco_val = [0, 5, 10]
efeito_tratamento_val = [0, 0, 0, 8]

ganho_peso = np.zeros(len(tratamento))
for i in range(3):  # Loop pelas gaiolas
    for j in range(4):  # Loop pelas dietas
        idx = i * 4 + j
        ganho_peso[idx] = 50 + efeito_bloco_val[i] + efeito_tratamento_val[j] + np.random.normal(0, 1)

dados_rcbd = pd.DataFrame({
    'Ganho_Peso': ganho_peso,
    'Tratamento': tratamento,
    'Bloco': bloco
})

# Modelo RCBD
modelo_rcbd = ols('Ganho_Peso ~ C(Tratamento) + C(Bloco)', data=dados_rcbd).fit()
print('ANOVA RCBD:')
print(sm.stats.anova_lm(modelo_rcbd, typ=2))

# Modelo CRD (ignorando blocos)
modelo_crd = ols('Ganho_Peso ~ C(Tratamento)', data=dados_rcbd).fit()
print('ANOVA CRD (Ignorando Blocos):')
print(sm.stats.anova_lm(modelo_crd, typ=2))

## 2.3 Exercício 3: ANOVA Fatorial Completa com Interação Forte

**Cenário:**
Produtividade em função de 2 níveis de Temperatura (T1, T2) e 2 níveis de Irrigação (I1, I2), com $n = 5$ repetições. Interação forte: $\mu_{T1,I2} \gg \mu_{T2,I2}$ (inversão do efeito).

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

np.random.seed(789)
n_rep = 5

Temperatura = np.repeat(['T1', 'T2'], n_rep * 2)
Irrigacao = np.tile(np.repeat(['I1', 'I2'], n_rep), 2)

# Médias: T1 I1: 100, T1 I2: 120; T2 I1: 110, T2 I2: 95
mu_vector_int = np.array(
    [100] * n_rep + [120] * n_rep +
    [110] * n_rep + [95] * n_rep
)

Produtividade = mu_vector_int + np.random.normal(0, 3, size=len(mu_vector_int))
dados_fatorial = pd.DataFrame({
    'Y': Produtividade,
    'Temperatura': Temperatura,
    'Irrigacao': Irrigacao
})

# Ajuste do Modelo Fatorial (Tipo II)
modelo_fatorial = ols('Y ~ C(Temperatura) * C(Irrigacao)', data=dados_fatorial).fit()
anova_table_fatorial = anova_lm(modelo_fatorial, typ=2)
print('Tabela ANOVA Fatorial:')
print(anova_table_fatorial)

# Preparação para Tukey HSD (Combinação dos Grupos)
dados_fatorial['Grupo'] = dados_fatorial['Temperatura'].astype(str) + ' x ' + dados_fatorial['Irrigacao'].astype(str)

# Aplicação do Tukey HSD nas 4 combinações
tukey_fatorial = pairwise_tukeyhsd(
    endog=dados_fatorial['Y'],
    groups=dados_fatorial['Grupo'],
    alpha=0.05
)
print("\nTeste Post-Hoc de Tukey HSD (Combinações):")
print(tukey_fatorial)

# Parte III: Análise de Resultados Esperados e Conclusões

## 3.1 Análise do TCL em Distribuição Assimétrica
A simulação da Parte I deverá produzir uma taxa de cobertura do IC próxima de 0,95.

1. **Efeito da Assimetria:** A população Exponencial original é fortemente assimétrica (skewness ≈ 2).
2. **Validação do TCL:** A distribuição das médias amostrais para $n = 50$ deve ter sua assimetria próxima de zero, e a forma do histograma deve ser aproximadamente Normal. O TCL garante que, para um $n$ grande o suficiente, a distribuição amostral se torna normal.
3. **Validação do IC:** A taxa de cobertura do IC de 95% é mantida (≈ 0,95), comprovando que a normalidade da distribuição das médias amostrais (via TCL) permite o uso confiável do método do IC baseado na distribuição t ou z, independentemente da distribuição populacional.

## 3.2 Análise da ANOVA de Planejamento Experimental
### 3.2.1 Exercício 2: RCBD (Blocagem)
A comparação entre a ANOVA RCBD e CRD ilustrará o ganho de poder estatístico.

- **Efeito do Bloco:** O bloco (gaiola) terá um p-valor muito baixo, indicando que a variação entre as gaiolas é significativa.
- **Erro Residual (MSResiduo):** O valor do MSResiduo será significativamente menor no modelo RCBD do que no modelo CRD.
- **Tratamento (Dieta):** O p-valor da Dieta será menor no modelo RCBD, confirmando a rejeição de $H_0$ de forma mais robusta, pois a variação sistemática do bloco foi isolada do erro aleatório.

### 3.2.2 Exercício 3: ANOVA Fatorial (Interação)
- **Interação T × I:** O p-valor para o termo de interação (Temperatura:Irrigação) será significativamente baixo ($p \ll 0{,}05$), indicando que o efeito da Irrigação depende criticamente da Temperatura.
- **Tukey HSD:** A análise do Tukey HSD nas 4 combinações (T1 x I1, T1 x I2, T2 x I1, T2 x I2) confirmará o padrão de inversão: a Irrigação Máxima (I2) é a melhor escolha apenas na Temperatura Baixa (T1), sendo uma das piores na Temperatura Alta (T2).
- **Conclusão:** A significância da interação impede a interpretação isolada dos efeitos principais, exigindo uma recomendação baseada na combinação ótima (T1 x I2).